In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor


DATA_PATH = "최종_용인_실거래가_통합_결측채움.csv"
MODEL_PATH = "xgb_yongin_price_model.joblib"
METRICS_PATH = "xgb_yongin_price_metrics.json"

USE_WANDB = False
RANDOM_STATE = 42
TEST_SIZE = 0.2
MIN_CATEGORY_COUNT = 50

COL_ADDRESS = "도로명"
COL_SIGUNGU = "시군구"
COL_AREA = "연면적(㎡)"
COL_LAND = "대지면적(㎡)"
COL_TYPE = "주택유형"
COL_YEAR = "건축년도"
COL_PRICE = "거래금액(만원)"
COL_TRADE_YM = "계약년월"
COL_TRADE_D = "계약일"
COL_ROAD_COND = "도로조건"


def normalize_money(value):
    text = str(value).replace(",", "").strip()
    return pd.to_numeric(text, errors="coerce")


def clean_category(value):
    text = str(value).strip()
    if text in {"", "-", "nan", "None"}:
        return "기타"
    return text


def extract_gu(addr):
    text = str(addr)
    for gu in ["수지구", "기흥구", "처인구"]:
        if gu in text:
            return gu
    return "기타"


def extract_eup_myeon_dong(addr):
    parts = str(addr).split()
    if not parts:
        return "기타"
    return parts[-1]


def prepare_dataframe(df):
    required = [
        COL_SIGUNGU,
        COL_AREA,
        COL_LAND,
        COL_TYPE,
        COL_YEAR,
        COL_PRICE,
        COL_TRADE_YM,
        COL_TRADE_D,
        COL_ROAD_COND,
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"필수 컬럼이 없습니다: {missing}")

    out = df.copy()

    out["contract_ym"] = pd.to_numeric(out[COL_TRADE_YM], errors="coerce")
    out["trade_year"] = (out["contract_ym"] // 100).astype("Int64")
    out["trade_month"] = (out["contract_ym"] % 100).astype("Int64")
    out["trade_day"] = pd.to_numeric(out[COL_TRADE_D], errors="coerce")

    out["price"] = out[COL_PRICE].apply(normalize_money)
    out["total_area"] = pd.to_numeric(out[COL_AREA], errors="coerce")
    out["land_area"] = pd.to_numeric(out[COL_LAND], errors="coerce")
    out["build_year"] = pd.to_numeric(out[COL_YEAR], errors="coerce")

    out = out.dropna(
        subset=[
            "price",
            "total_area",
            "land_area",
            "build_year",
            "trade_year",
            "trade_month",
            "trade_day",
        ]
    )

    out = out[
        (out["price"] > 0)
        & (out["total_area"] > 0)
        & (out["land_area"] > 0)
    ]

    out["trade_year"] = out["trade_year"].astype(int)
    out["trade_month"] = out["trade_month"].astype(int)
    out["trade_day"] = out["trade_day"].astype(int)
    out["build_year"] = out["build_year"].astype(int)

    out["building_age"] = out["trade_year"] - out["build_year"]
    out = out[(out["building_age"] >= 0) & (out["building_age"] <= 150)]

    out["far"] = out["total_area"] / out["land_area"]
    out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=["far"])
    out = out[(out["far"] > 0) & (out["far"] < 20)]

    out["log_total_area"] = np.log1p(out["total_area"])
    out["log_land_area"] = np.log1p(out["land_area"])

    out["house_type"] = out[COL_TYPE].map(clean_category)
    out["gu"] = out[COL_SIGUNGU].apply(extract_gu)

    out["dong"] = (
        out[COL_SIGUNGU]
        .apply(extract_eup_myeon_dong)
        .map(clean_category)
    )

    out["road_condition"] = out[COL_ROAD_COND].map(clean_category)

    out["contract_date_order"] = (
        out["trade_year"] * 10000
        + out["trade_month"] * 100
        + out["trade_day"]
    )

    price_lower = out["price"].quantile(0.01)
    price_upper = out["price"].quantile(0.99)

    out = out[
        (out["price"] >= price_lower)
        & (out["price"] <= price_upper)
    ]

    out = out.sort_values("contract_date_order").reset_index(drop=True)

    return out


def time_based_split(df, test_size=0.2):
    split_idx = int(len(df) * (1 - test_size))
    split_idx = min(max(split_idx, 1), len(df) - 1)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()


class DongTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column="dong", smoothing=10):
        self.column = column
        self.smoothing = smoothing

    def fit(self, X, y, sample_weight=None):
        X_temp = X.copy()
        X_temp["_target"] = y.values

        stats = X_temp.groupby(self.column)["_target"].agg(["mean", "count"])

        self.global_mean_ = y.mean()

        self.mapping_ = (
            (stats["mean"] * stats["count"] + self.global_mean_ * self.smoothing)
            / (stats["count"] + self.smoothing)
        )

        return self

    def transform(self, X):
        X_out = X.copy()

        X_out["dong_avg_price"] = (
            X_out[self.column]
            .map(self.mapping_)
            .fillna(self.global_mean_)
        )

        return X_out


NUM_FEATURES = [
    "total_area",
    "land_area",
    "log_total_area",
    "log_land_area",
    "far",
    "building_age",
    "build_year",
    "trade_year",
    "trade_month",
    "trade_day",
    "dong_avg_price",
]

CAT_FEATURES = [
    "gu",
    "dong",
    "road_condition",
    "house_type",
]

BASE_FEATURES = [
    c for c in NUM_FEATURES
    if c != "dong_avg_price"
] + CAT_FEATURES


def build_pipeline(params):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), NUM_FEATURES),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                min_frequency=MIN_CATEGORY_COUNT,
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                CAT_FEATURES,
            ),
        ]
    )

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        **params,
    )

    return Pipeline(
        steps=[
            ("target_encoder", DongTargetEncoder(column="dong", smoothing=10)),
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )


def evaluate(y_true, pred):
    mae = mean_absolute_error(y_true, pred)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    r2 = r2_score(y_true, pred)

    y_true_array = np.asarray(y_true, dtype=float)
    pred_array = np.asarray(pred, dtype=float)

    nonzero_mask = y_true_array != 0

    mape = (
        np.mean(
            np.abs(
                (y_true_array[nonzero_mask] - pred_array[nonzero_mask])
                / y_true_array[nonzero_mask]
            )
        )
        * 100
        if nonzero_mask.any()
        else np.nan
    )

    wmape = (
        np.sum(np.abs(y_true_array - pred_array))
        / np.sum(np.abs(y_true_array))
        * 100
        if np.sum(np.abs(y_true_array)) != 0
        else np.nan
    )

    return {
        "MAE_만원": float(mae),
        "RMSE_만원": float(rmse),
        "R2": float(r2),
        "MAPE_%": float(mape),
        "WMAPE_%": float(wmape),
    }


def analyze_jeonse_risk(
    estimated_price_manwon,
    mortgage_amount_manwon,
    deposit_manwon,
    liquidation_rate=0.85,
):
    if estimated_price_manwon <= 0:
        raise ValueError("estimated_price_manwon must be positive")

    mortgage_ltv = mortgage_amount_manwon / estimated_price_manwon
    total_ltv = (mortgage_amount_manwon + deposit_manwon) / estimated_price_manwon

    expected_liquidation_value = estimated_price_manwon * liquidation_rate
    recoverable_deposit = max(0, expected_liquidation_value - mortgage_amount_manwon)
    recoverable_deposit = min(deposit_manwon, recoverable_deposit)

    expected_loss = max(0, deposit_manwon - recoverable_deposit)

    recovery_rate = (
        recoverable_deposit / deposit_manwon
        if deposit_manwon > 0
        else 1.0
    )

    if mortgage_ltv <= 0.50 and total_ltv <= 0.70 and recovery_rate >= 0.95:
        grade = "안전"
    elif mortgage_ltv <= 0.70 and total_ltv <= 0.90 and recovery_rate >= 0.80:
        grade = "주의"
    else:
        grade = "위험"

    return {
        "estimated_price_manwon": float(estimated_price_manwon),
        "mortgage_amount_manwon": float(mortgage_amount_manwon),
        "deposit_manwon": float(deposit_manwon),
        "mortgage_ltv": float(mortgage_ltv),
        "total_ltv": float(total_ltv),
        "liquidation_rate": float(liquidation_rate),
        "recoverable_deposit_manwon": float(recoverable_deposit),
        "expected_loss_manwon": float(expected_loss),
        "deposit_recovery_rate": float(recovery_rate),
        "safety_grade": grade,
    }


def main():
    if USE_WANDB:
        import wandb
        wandb.login()

    df = pd.read_csv(DATA_PATH)
    data = prepare_dataframe(df)

    train_df, test_df = time_based_split(data, test_size=TEST_SIZE)

    X_train = train_df[BASE_FEATURES]
    y_train = train_df["price"]

    X_test = test_df[BASE_FEATURES]
    y_test = test_df["price"]

    weight_base_year = train_df["trade_year"].min()

    sample_weight = 0.7 + 0.3 * (
        (train_df["trade_year"] - weight_base_year)
        / max(1, train_df["trade_year"].max() - weight_base_year)
    )

    param_grid = [
        {
            "n_estimators": 300,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.0,
        },
        {
            "n_estimators": 500,
            "max_depth": 4,
            "learning_rate": 0.03,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.0,
        },
        {
            "n_estimators": 500,
            "max_depth": 5,
            "learning_rate": 0.03,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_lambda": 2.0,
        },
        {
            "n_estimators": 700,
            "max_depth": 5,
            "learning_rate": 0.02,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_lambda": 2.0,
        },
    ]

    best_score = -np.inf
    best_params = None

    cv = TimeSeriesSplit(n_splits=5)

    for params in param_grid:
        pipeline = build_pipeline(params)
        cv_scores = []

        for train_idx, valid_idx in cv.split(X_train):
            X_cv_train = X_train.iloc[train_idx]
            y_cv_train = y_train.iloc[train_idx]

            X_cv_valid = X_train.iloc[valid_idx]
            y_cv_valid = y_train.iloc[valid_idx]

            w_cv_train = sample_weight.iloc[train_idx]

            pipeline.fit(
                X_cv_train,
                y_cv_train,
                target_encoder__sample_weight=w_cv_train,
                model__sample_weight=w_cv_train,
            )

            cv_pred = pipeline.predict(X_cv_valid)
            cv_score = -mean_absolute_error(y_cv_valid, cv_pred)

            cv_scores.append(cv_score)

        score = np.mean(cv_scores)

        print(f"params={params} | CV MAE={-score:.4f}")

        if USE_WANDB:
            import wandb

            with wandb.init(
                project="yongin-ai",
                entity="jk09145123456-dankook-university",
                name=f"XGB_{params}",
                config=params,
                reinit=True,
            ):
                wandb.log({"cv_mae": float(-score)})

        if score > best_score:
            best_score = score
            best_params = params

    best_pipeline = build_pipeline(best_params)

    best_pipeline.fit(
        X_train,
        y_train,
        target_encoder__sample_weight=sample_weight,
        model__sample_weight=sample_weight,
    )

    pred = best_pipeline.predict(X_test)
    metrics = evaluate(y_test, pred)

    sample_estimated_price = float(pred[0])

    sample_risk = analyze_jeonse_risk(
        estimated_price_manwon=sample_estimated_price,
        mortgage_amount_manwon=sample_estimated_price * 0.45,
        deposit_manwon=sample_estimated_price * 0.25,
    )

    print("\n===== Final Test Metrics =====")
    print(f"rows_total: {len(data):,}")
    print(f"train_rows: {len(train_df):,}")
    print(f"test_rows: {len(test_df):,}")
    print(f"test_period: {test_df['contract_ym'].min()} ~ {test_df['contract_ym'].max()}")
    print(f"best_params: {best_params}")

    for k, v in metrics.items():
        print(f"{k}: {v:,.4f}" if isinstance(v, float) else f"{k}: {v}")

    print("\n===== Jeonse Risk Example =====")
    print(f"estimated_price: {sample_risk['estimated_price_manwon']:,.0f} 만원")
    print(f"mortgage_amount: {sample_risk['mortgage_amount_manwon']:,.0f} 만원")
    print(f"deposit: {sample_risk['deposit_manwon']:,.0f} 만원")
    print(f"mortgage_ltv: {sample_risk['mortgage_ltv']:.2%}")
    print(f"total_ltv: {sample_risk['total_ltv']:.2%}")
    print(f"recoverable_deposit: {sample_risk['recoverable_deposit_manwon']:,.0f} 만원")
    print(f"expected_loss: {sample_risk['expected_loss_manwon']:,.0f} 만원")
    print(f"safety_grade: {sample_risk['safety_grade']}")

    result = {
        "rows_total": int(len(data)),
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "test_period": [
            int(test_df["contract_ym"].min()),
            int(test_df["contract_ym"].max()),
        ],
        "best_params": best_params,
        "metrics": metrics,
        "risk_example": sample_risk,
        "num_features": NUM_FEATURES,
        "cat_features": CAT_FEATURES,
        "base_features": BASE_FEATURES,
    }

    joblib.dump(
        {
            "pipeline": best_pipeline,
            "num_features": NUM_FEATURES,
            "cat_features": CAT_FEATURES,
            "base_features": BASE_FEATURES,
            "metrics": result,
        },
        MODEL_PATH,
    )

    Path(METRICS_PATH).write_text(
        json.dumps(result, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"\nmodel saved: {MODEL_PATH}")
    print(f"metrics saved: {METRICS_PATH}")


if __name__ == "__main__":
    main()